In [ ]:
import cv2
import imutils
import datetime
from google.colab.patches import cv2_imshow

In [ ]:
image_url = "https://raw.githubusercontent.com/Morteza-Asadi-Shalmaiy/Motion-Detection/refs/heads/main/assets/test-video.mp4"
image_path = "/content/test-video.mp4"

# geting the image using wget
!wget -q $image_url -O $image_path
!ls

In [ ]:
video_path = "/content/test-video.mp4"  # input video path
min_area = 500

In [ ]:
vs = cv2.VideoCapture(video_path)
firstFrame = None

In [ ]:
import os

# derive output name: result-<original filename>
input_filename = os.path.basename(video_path)
output_path = f"/content/result-{input_filename}"

total_frames = int(vs.get(cv2.CAP_PROP_FRAME_COUNT))
fps = vs.get(cv2.CAP_PROP_FPS) or 20  # fallback if fps metadata is missing

writer = None
frame_idx = 0
motion_frame_count = 0

while True:
    ret, frame = vs.read()
    if not ret:
        break

    frame_idx += 1
    text = "No motion"

    frame = imutils.resize(frame, width=500)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)

    if firstFrame is None:
        firstFrame = gray
        # still write the very first frame so output length matches input
        if writer is None:
            h, w = frame.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
        writer.write(frame)
        print(f"\rProcessing frame {frame_idx}/{total_frames} - initializing background", end="")
        continue

    frameDelta = cv2.absdiff(firstFrame, gray)
    thresh = cv2.threshold(frameDelta, 25, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.dilate(thresh, None, iterations=2)
    cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for c in cnts:
        if cv2.contourArea(c) < min_area:
            continue
        (x, y, w, h) = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        text = "Motion detected"

    if text == "Motion detected":
        motion_frame_count += 1

    cv2.putText(frame, f"Room Status: {text}", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    cv2.putText(frame, datetime.datetime.now().strftime("%A %d %B %Y %I:%M:%S%p"),
                (10, frame.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)

    if writer is None:
        h, w = frame.shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    writer.write(frame)

    # single updating progress line, no per-frame image spam
    print(f"\rProcessing frame {frame_idx}/{total_frames} - {text}", end="")

vs.release()
if writer is not None:
    writer.release()

print(f"\nDone. {motion_frame_count}/{frame_idx} frames had motion.")
print(f"Saved annotated video to: {output_path}")

In [ ]:
!pip install -q imageio imageio-ffmpeg
!pip install imageio[pyav]

In [ ]:
output_path = "/content/result-test-video.mp4"

In [ ]:
import imageio.v3 as iio

def make_gif_preview(video_path, gif_path, seconds=15, fps=10, resize_width=400):
    reader = iio.imiter(video_path, plugin="pyav")
    frames = []
    frame_interval = None
    src_fps = None

    meta = iio.immeta(video_path, plugin="pyav")
    src_fps = meta.get("fps", 20)
    frame_interval = max(1, round(src_fps / fps))
    max_frames = int(seconds * src_fps)

    for i, frame in enumerate(reader):
        if i >= max_frames:
            break
        if i % frame_interval != 0:
            continue
        h, w = frame.shape[:2]
        new_h = int(h * (resize_width / w))
        frame = cv2.resize(frame, (resize_width, new_h))
        frames.append(frame)

    iio.imwrite(gif_path, frames, duration=1000 / fps, loop=0)
    print(f"Saved {seconds}s GIF preview to: {gif_path}")

gif_path = output_path.replace(".mp4", ".gif")
make_gif_preview(output_path, gif_path, seconds=15, fps=10, resize_width=400)

# preview it inline in Colab
from IPython.display import Image
Image(open(gif_path, "rb").read())